# HPO TÜV

Wiederholt einen SolarSystemLander-Trial mit denselben HPs und demselben Seed. Ähnliche Ergebnisse sprechen dafür, dass die Elise reproduzierbar arbeitet.

## Setup

In [ ]:
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.colab import prepare_study_storage, setup_colab

COLAB = setup_colab()

## Check 1: Trial reproduzieren

Wiederholt einen SolarSystemLander-Trial mit denselben HPs und demselben Seed. Ähnliche Ergebnisse sprechen dafür, dass die Elise reproduzierbar arbeitet.

### Trial auswählen

In [ ]:
import optuna
import torch
from hpo.params import HP

OBSERVATION_MODE = "8d"
STUDY_SERIES_NAME = f"solar_system_lander_{OBSERVATION_MODE}_elise"
STUDY_NAME = "s1_flight_hours"
TRIAL_NUMBER = 1

BASE_PARAMS = {
    HP.LEARNING_RATE: 0.0022727854024196057,
    HP.BATCH_SIZE: 512,
    HP.EPS_END: 0.047716002108220544,
    HP.EPS_DECAY: 43_214,
    HP.GAMMA: 0.99,
    HP.TAU: 0.005,
    HP.LEARNING_STARTS: 2_500,
    HP.OPTIMIZE_EVERY: 2,
    HP.REPLAY_MEMORY_CAPACITY: 400_000,
    HP.NUM_EPISODES: 500,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STUDY_SERIES_STORAGE = prepare_study_storage(COLAB, STUDY_SERIES_NAME)
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=f"sqlite:///{STUDY_SERIES_STORAGE.database_path}",
)
trial = next(t for t in study.trials if t.number == TRIAL_NUMBER)
params = BASE_PARAMS | trial.params

print(f"Trial:          {trial.number}")
print(f"Original score: {trial.value:.1f}")
print(f"Training seed:  {trial.user_attrs['trial_seed']}")
print(f"Device:         {device}")
params

### Trial wiederholen

In [ ]:
from dqn.vector_training import VectorTrainer
from hpo.objective import evaluate_greedy_q_net, vector_training_config
from hpo.solar_system_lander.environment import EnvFactory

config = vector_training_config(params)
factory = EnvFactory(study.user_attrs["observation_mode"])
env = factory.make_training_env(study.user_attrs["num_envs"])
try:
    trainer = VectorTrainer(
        env,
        seed=trial.user_attrs["trial_seed"],
        device=device,
        replay_memory_capacity=params[HP.REPLAY_MEMORY_CAPACITY],
    )
    repeated = trainer.train(config)
finally:
    env.close()

eval_seeds = study.user_attrs["eval_seeds"]
repeated_world_scores = {
    name: evaluate_greedy_q_net(
        q_net=repeated.q_net,
        device=trainer.device,
        make_env=make_env,
        episodes=len(eval_seeds),
        max_steps=study.user_attrs["eval_max_steps"],
        seed=eval_seeds[0],
    )
    for name, make_env in factory.evaluation_envs().items()
}
repeated_score = sum(repeated_world_scores.values()) / len(repeated_world_scores)

### Vergleich

In [ ]:
import pandas as pd
import plotly.graph_objects as go

original_world_scores = trial.user_attrs["world_scores"]
scores = pd.DataFrame({
    "Original": original_world_scores | {"mean": trial.value},
    "Wiederholung": repeated_world_scores | {"mean": repeated_score},
})
scores["Differenz"] = scores["Wiederholung"] - scores["Original"]
display(scores.round(1))

curves = pd.DataFrame({
    "Original": trial.user_attrs["training_curve"]["episode_returns"],
    "Wiederholung": repeated.episode_returns,
}).rolling(100).mean()
display(px.line(
    curves,
    labels={"index": "Episode", "value": "Return", "variable": "Lauf"},
    title="Training · gleitender Mittelwert über 100 Episoden",
))

## Check 2: Single-World-Fähigkeit

Prüft als schnelles Indiz, ob der `VectorTrainer` auf einer einzelnen Welt einen brauchbaren Lander lernen kann.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
import torch
from gymnasium.vector import SyncVectorEnv
from IPython.display import clear_output, display
from dqn.vector_training import VectorTrainer, VectorTrainingConfig
from hpo.objective import evaluate_greedy_q_net
from hpo.solar_system_lander.environment import EnvFactory

EPISODES = 500
MIN_SCORE = 100
NUM_ENVS = 20
REPLAY_MEMORY_CAPACITY = 400_000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Progress:
    def __init__(self):
        self.started = perf_counter()
        self.last_update = self.started

    def plot_returns(self, returns, epsilons=None):
        completed = len(returns)
        now = perf_counter()
        if now - self.last_update < 5 and completed < EPISODES:
            return
        self.last_update = now
        elapsed = now - self.started
        remaining = elapsed / completed * (EPISODES - completed)
        curve = pd.DataFrame({"Return": returns})
        curve["Mean (50 episodes)"] = curve["Return"].rolling(50).mean()
        figure, axis = plt.subplots(figsize=(10, 5))
        axis.plot(curve["Return"], alpha=0.25, label="Return")
        axis.plot(curve["Mean (50 episodes)"], label="Mean (50 episodes)")
        axis.set_xlim(0, EPISODES)
        axis.set_xlabel("Episode")
        axis.set_ylabel("Return")
        axis.set_title(
            f"Single-World-Check · {completed}/{EPISODES} Episoden · "
            f"noch ca. {remaining / 60:.1f} min"
        )
        axis.legend()
        clear_output(wait=True)
        display(figure)
        plt.close(figure)

config = VectorTrainingConfig(
    num_episodes=EPISODES,
    batch_size=512,
    gamma=0.99,
    eps_start=1.0,
    eps_end=0.047716002108220544,
    eps_decay=43_214,
    tau=0.005,
    learning_rate=0.0022727854024196057,
    learning_starts=2_500,
    optimize_every=2,
)
factory = EnvFactory(OBSERVATION_MODE)
make_moon = factory.evaluation_envs()["moon"]
env = SyncVectorEnv([make_moon] * NUM_ENVS)
try:
    trainer = VectorTrainer(
        env,
        seed=42,
        device=device,
        replay_memory_capacity=REPLAY_MEMORY_CAPACITY,
    )
    result = trainer.train(
        config,
        plotter=Progress(),
    )
finally:
    env.close()

score = evaluate_greedy_q_net(
    q_net=result.q_net,
    device=trainer.device,
    make_env=make_moon,
    episodes=10,
    max_steps=2_000,
    seed=10_000,
)
print(f"Score: {score:.1f}")
print("PASS" if score >= MIN_SCORE else "CHECK")